In [3]:
"""
engine.py
────────────────────────────────────────────────────────────────────────────────
订单簿重建引擎（Order Book Reconstructor）

数据加载由外部 main.py / DataLoader 负责，OrderBook 只负责状态维护。

使用方式：
    from order_book import OrderBook

    ob = OrderBook()
    ob.init_from_snapshot(row)   # 传入一行 level2 快照（pd.Series）
    ob.apply_order(order_row)    # 传入一笔 limit 委托
    ob.apply_trade(trade_row)    # 传入一笔成交
    snapshot = ob.to_snapshot()  # 输出当前盘口状态（平铺 dict）
────────────────────────────────────────────────────────────────────────────────
"""

import pandas as pd

# 价格整数化倍数（规避浮点误差）
_PRICE_SCALE = 1000


def _to_int_price(price: float) -> int:
    """将浮点价格转换为整数 Key，避免 2.95 != 2.9500000001 的问题。"""
    return round(price * _PRICE_SCALE)


def _to_float_price(int_price: int) -> float:
    """将整数 Key 还原为浮点价格。"""
    return int_price / _PRICE_SCALE


class OrderBook:
    """
    基于 Level2 快照 + 逐笔委托/成交流水，在内存中实时重建订单簿。
    数据加载由外部完成，本类只负责状态维护与输出。
    """

    LEVELS = 10  # 输出档位数

    # ── 初始化 ────────────────────────────────────────────────────────────────
    def __init__(self):
        # 核心盘口容器：{int_price: volume}
        self.bids: dict[int, int] = {}
        self.asks: dict[int, int] = {}

        # 元数据状态
        self.last_price: float = 0.0
        self.cum_volume: int   = 0
        self.cum_amount: float = 0.0
        self.timestamp:  int   = 0

        # 买卖一档缓存（整数价格，避免重复转换）
        self._bp1_int: int = 0
        self._ap1_int: int = 0
        self.bv1:      int = 0
        self.av1:      int = 0

        # 衍生指标
        self.spread:    float = 0.0
        self.mid_price: float = 0.0
        self.imbalance: float = 0.0

    # ── 只读属性：对外暴露浮点价格 ───────────────────────────────────────────
    @property
    def bp1(self) -> float:
        return _to_float_price(self._bp1_int)

    @property
    def ap1(self) -> float:
        return _to_float_price(self._ap1_int)

    # ── 衍生指标计算 ──────────────────────────────────────────────────────────
    def _update_metrics(self) -> None:
        """根据当前 bp1/ap1/bv1/av1 实时更新 spread、mid_price、imbalance。
        当某一边盘口为空（价格 = 0）时，spread 和 mid_price 置为 NaN，
        避免出现负数价差或以 0 参与计算的无意义结果。
        """
        bp = self.bp1   # 买盘空时为 0.0
        ap = self.ap1   # 卖盘空时为 0.0

        bid_empty = (self._bp1_int == 0)
        ask_empty = (self._ap1_int == 0)

        if bid_empty or ask_empty:
            self.spread    = float("nan")
            self.mid_price = float("nan")
        else:
            self.spread    = round(ap - bp, 6)
            self.mid_price = round((bp + ap) / 2, 6)

        total = self.bv1 + self.av1
        self.imbalance = round((self.bv1 - self.av1) / total, 6) if total > 0 else 0.0

    # ── 1. 从 Level2 快照初始化 ───────────────────────────────────────────────
    def init_from_snapshot(self, row: pd.Series) -> None:
        """
        将一行 Level2 快照解压到内存盘口。

        Parameters
        ----------
        row : pd.Series
            df_level2 中的一行（index 为 time）。
        """
        # 清空残留数据
        self.bids.clear()
        self.asks.clear()

        # 装载买盘 bp1-bp10 / bv1-bv10
        for i in range(1, self.LEVELS + 1):
            price = float(row[f"bp{i}"])
            vol   = int(row[f"bv{i}"])
            if vol > 0 and price > 0:
                self.bids[_to_int_price(price)] = vol

        # 装载卖盘 ap1-ap10 / av1-av10
        for i in range(1, self.LEVELS + 1):
            price = float(row[f"ap{i}"])
            vol   = int(row[f"av{i}"])
            if vol > 0 and price > 0:
                self.asks[_to_int_price(price)] = vol

        # 元数据
        self.last_price = float(row["last"])
        self.cum_volume = int(row["volume"])
        self.cum_amount = float(row["amount"])
        self.timestamp  = int(row.name)  # index = time

        # 买卖一档缓存（直接用整数存储）
        self._bp1_int = _to_int_price(float(row["bp1"]))
        self._ap1_int = _to_int_price(float(row["ap1"]))
        self.bv1      = int(row["bv1"])
        self.av1      = int(row["av1"])

        self._update_metrics()

    # ── 2. 处理逐笔 limit 委托 ────────────────────────────────────────────────
    def apply_order(self, row) -> None:
        """
        将一笔 limit 委托更新到内存盘口。
        market / stop 单跳过不处理。

        Parameters
        ----------
        row : namedtuple
            itertuples() 返回的行对象。
        """
        if str(getattr(row, "order_type")) != "limit":
            return

        side     = str(getattr(row, "side"))
        int_p    = _to_int_price(float(getattr(row, "price")))
        quantity = int(getattr(row, "quantity"))

        if side == "buy":
            self.bids[int_p] = self.bids.get(int_p, 0) + quantity
            # 缓存维护：只有影响到买一才更新
            if int_p > self._bp1_int:
                self._bp1_int = int_p
                self.bv1      = quantity
                self._update_metrics()
            elif int_p == self._bp1_int:
                self.bv1 += quantity
                self._update_metrics()
        else:
            self.asks[int_p] = self.asks.get(int_p, 0) + quantity
            # 缓存维护：只有影响到卖一才更新
            # _ap1_int == 0 表示卖盘原本为空，任何新卖单都是新的卖一
            if self._ap1_int == 0 or int_p < self._ap1_int:
                self._ap1_int = int_p
                self.av1      = quantity
                self._update_metrics()
            elif int_p == self._ap1_int:
                self.av1 += quantity
                self._update_metrics()

        self.timestamp = int(row.name)

    # ── 3. 处理逐笔成交 ───────────────────────────────────────────────────────
    def apply_trade(self, row) -> None:
        """
        将一笔成交从对应盘口扣减挂单量。
        只有当成交真正吃掉了买一/卖一时，才触发 max/min 全盘扫描。

        Parameters
        ----------
        row : namedtuple
            itertuples() 返回的行对象。
        """
        price    = float(getattr(row, "price"))
        quantity = int(getattr(row, "quantity"))
        int_p    = _to_int_price(price)

        # 因果律判定：确定扣减方向
        if int_p <= self._bp1_int:
            book        = self.bids
            hit_top     = (int_p == self._bp1_int)
            is_bid_side = True
        elif int_p >= self._ap1_int:
            book        = self.asks
            hit_top     = (int_p == self._ap1_int)
            is_bid_side = False
        else:
            # 穿透效应：价格落在 bp1 与 ap1 之间，归类到更近的一侧
            # 若某一边为空（价格 = 0），该边距离视为无穷大，归到另一边
            bid_empty = (self._bp1_int == 0)
            ask_empty = (self._ap1_int == 0)

            if bid_empty and ask_empty:
                return  # 双边均空，无法处理，静默跳过
            elif bid_empty:
                book        = self.asks
                hit_top     = (int_p == self._ap1_int)
                is_bid_side = False
            elif ask_empty:
                book        = self.bids
                hit_top     = (int_p == self._bp1_int)
                is_bid_side = True
            else:
                dist_bid = int_p - self._bp1_int   # 到买一的距离（正数）
                dist_ask = self._ap1_int - int_p   # 到卖一的距离（正数）
                if dist_bid <= dist_ask:
                    book        = self.bids
                    hit_top     = (int_p == self._bp1_int)
                    is_bid_side = True
                else:
                    book        = self.asks
                    hit_top     = (int_p == self._ap1_int)
                    is_bid_side = False

        # 挂单量扣减（防御负数）
        top_cleared = False
        if int_p in book:
            book[int_p] = max(0, book[int_p] - quantity)
            if book[int_p] == 0:
                del book[int_p]
                top_cleared = hit_top   # 只有买一/卖一被完全清空才需要重新扫描

        # 全局成交统计
        self.last_price  = price
        self.cum_volume += quantity
        self.cum_amount += price * quantity

        # ── 缓存维护：只有 Top-of-Book 被清空才做全盘扫描 ─────────────────────
        if is_bid_side:
            if top_cleared:
                # 买一被完全吃掉 → 重新扫描找新的最高买价
                if self.bids:
                    self._bp1_int = max(self.bids)
                    self.bv1      = self.bids[self._bp1_int]
                else:
                    self._bp1_int = 0
                    self.bv1      = 0
            elif hit_top:
                # 买一还在，但量减少了 → 直接读字典，无需扫描
                self.bv1 = self.bids.get(self._bp1_int, 0)
        else:
            if top_cleared:
                # 卖一被完全吃掉 → 重新扫描找新的最低卖价
                if self.asks:
                    self._ap1_int = min(self.asks)
                    self.av1      = self.asks[self._ap1_int]
                else:
                    self._ap1_int = 0
                    self.av1      = 0
            elif hit_top:
                # 卖一还在，但量减少了 → 直接读字典，无需扫描
                self.av1 = self.asks.get(self._ap1_int, 0)

        self._update_metrics()
        self.timestamp = int(row.name)

    # ── 4. 输出当前盘口快照 ───────────────────────────────────────────────────
    def to_snapshot(self) -> dict:
        """
        将当前内存盘口归一化，输出平铺字典。

        Returns
        -------
        dict
            包含 bp1-bp10、bv1-bv10、ap1-ap10、av1-av10、
            spread、mid_price、imbalance、last_price、
            cum_volume、cum_amount、timestamp。
        """
        # 买盘：降序取前10档
        sorted_bids = sorted(self.bids.keys(), reverse=True)[:self.LEVELS]
        # 卖盘：升序取前10档
        sorted_asks = sorted(self.asks.keys())[:self.LEVELS]

        result: dict = {
            "timestamp":  self.timestamp,
            "last_price": self.last_price,
            "cum_volume": self.cum_volume,
            "cum_amount": round(self.cum_amount, 2),
            "spread":     self.spread,
            "mid_price":  self.mid_price,
            "imbalance":  self.imbalance,
        }

        # 买盘档位（不足10档用 0 补齐）
        for i in range(self.LEVELS):
            if i < len(sorted_bids):
                int_p = sorted_bids[i]
                result[f"bp{i+1}"] = _to_float_price(int_p)
                result[f"bv{i+1}"] = self.bids[int_p]
            else:
                result[f"bp{i+1}"] = 0.0
                result[f"bv{i+1}"] = 0

        # 卖盘档位（不足10档用 0 补齐）
        for i in range(self.LEVELS):
            if i < len(sorted_asks):
                int_p = sorted_asks[i]
                result[f"ap{i+1}"] = _to_float_price(int_p)
                result[f"av{i+1}"] = self.asks[int_p]
            else:
                result[f"ap{i+1}"] = 0.0
                result[f"av{i+1}"] = 0

        return result